# Orquestrador do pipeline Libellula

Este notebook automatiza a reconstrução do dataset técnico, das cinco ferramentas do Envelope de Risco, das previsões do TFT e do estado probabilístico de 29 features (25 de risco + 4 preditivas). Para executar todo o fluxo, selecione **Restart Kernel and Run All**.

A ordem é: **Financial Tools → CVaR → EPF → EVT → Quantile Forecasting → PDFD → TFT → concatenação do risco → rl_state**. EPF precede obrigatoriamente EVT; as concatenações são executadas somente após seus insumos. Os caminhos abaixo correspondem aos arquivos existentes no projeto.

Cada notebook roda integralmente em um processo Python isolado, com o diretório de trabalho do próprio arquivo. Isso preserva caminhos relativos e evita que variáveis de uma etapa contaminem outra. Os arquivos de código originais não são reescritos. As saídas de dados e modelos são atualizadas pelas rotinas existentes, incluindo o treinamento completo do TFT.

Cada execução cria logs e um relatório JSON em `data/processed/pipeline_runs/`. Uma falha interrompe o fluxo e mostra o log correspondente; não há rollback dos artefatos já produzidos. Após corrigir a causa, reinicie o kernel e execute tudo novamente. Notebooks auxiliares de setup, auditoria e Monte Carlo não integram esta sequência.


## Configuração e executor

Localiza a raiz a partir do diretório atual, seleciona o Python do ambiente `.venv` quando disponível, verifica antecipadamente os arquivos e dependências e cria o registro da execução. O executor respeita a ordem declarada, registra duração e status e propaga erros. `STEP_TIMEOUT_SECONDS` limita cada etapa a duas horas e pode ser ajustado para treinamentos maiores. As saídas completas ficam nos logs; esta tela apresenta início e conclusão de cada etapa.


In [1]:
from pathlib import Path
from datetime import datetime
import json
import os
import subprocess
import sys
import time
import uuid

PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "src/pipelines/rl_state.py").is_file()
     and (p / "notebooks/Predictive_TFT_model/TFT_prdictive_model.ipynb").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Abra este notebook em um diretório dentro do projeto Libellula.")

venv_python = PROJECT_ROOT / ".venv" / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
PYTHON_EXECUTABLE = str(venv_python if venv_python.is_file() else Path(sys.executable))
STEP_TIMEOUT_SECONDS = 7200
STEPS = [
    ("01_financial_tools", "notebooks/financial_tools/financial_tools.ipynb"),
    ("02_cvar", "notebooks/risk_envelop/CVaR.ipynb"),
    ("03_epf", "notebooks/risk_envelop/EPF.ipynb"),
    ("04_evt", "notebooks/risk_envelop/EVT.ipynb"),
    ("05_quantile_forecasting", "notebooks/risk_envelop/QuantileForecasting.ipynb"),
    ("06_pdfd", "notebooks/risk_envelop/PDFD.ipynb"),
    ("07_tft", "notebooks/Predictive_TFT_model/TFT_prdictive_model.ipynb"),
    ("08_concatenation", "notebooks/risk_envelop/concatenation/Concatenation.ipynb"),
    ("09_rl_state", "src/pipelines/rl_state.py"),
]
for _, relative_path in STEPS:
    if not (PROJECT_ROOT / relative_path).is_file():
        raise FileNotFoundError(PROJECT_ROOT / relative_path)

# Verificar no mesmo interpretador que efetivamente executará as etapas.
preflight = subprocess.run(
    [PYTHON_EXECUTABLE, "-c",
     "import importlib.util; names = ['IPython', 'pandas', 'numpy', 'torch', "
     "'pytorch_forecasting', 'lightning', 'pytorch_lightning', 'matplotlib', "
     "'pyarrow', 'sklearn', 'scipy', 'joblib']; "
     "missing = [n for n in names if importlib.util.find_spec(n) is None]; "
     "assert not missing, f'Dependências ausentes: {missing}'"],
    capture_output=True, text=True, encoding="utf-8", errors="replace", check=False,
)
if preflight.returncode:
    raise RuntimeError(preflight.stderr or preflight.stdout)

RUN_DIR = PROJECT_ROOT / "data/processed/pipeline_runs" / (
    datetime.now().strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:8]
)
RUN_DIR.mkdir(parents=True, exist_ok=False)
REPORT_PATH = RUN_DIR / "status.json"
results = []

# IPython executa células com magics e comandos !, sem exigir nbconvert/nbclient.
# Cada processo conserva apenas o namespace do notebook que está executando.
NOTEBOOK_RUNNER = r"""
import json
from pathlib import Path
import sys
from IPython.core.interactiveshell import InteractiveShell

path = Path(sys.argv[1])
notebook = json.loads(path.read_text(encoding="utf-8"))
shell = InteractiveShell.instance()
for index, cell in enumerate(notebook["cells"]):
    if cell["cell_type"] != "code":
        continue
    source = cell["source"]
    if isinstance(source, list):
        source = "".join(source)
    print(f"\n--- {path.name}: célula {index + 1} ---", flush=True)
    result = shell.run_cell(source, store_history=False)
    if result.error_before_exec is not None or result.error_in_exec is not None:
        raise SystemExit(1)
"""
STATE_RUNNER = r"""
from pathlib import Path
import runpy
import sys

root = Path(sys.argv[1])
module = runpy.run_path(str(root / "src/pipelines/rl_state.py"))
state = module["build_rl_state"](
    risk_path=root / "data/processed/combined/risk_features.parquet",
    prediction_path=root / "data/processed/predictions/tft_predictions.parquet",
    output_path=root / "data/processed/rl_state/rl_state.parquet",
)
print(f"Estado gerado: {state.shape[0]} linhas, {state.shape[1]} colunas.")
"""

def save_report():
    REPORT_PATH.write_text(json.dumps({
        "python": PYTHON_EXECUTABLE,
        "project_root": str(PROJECT_ROOT),
        "planned_steps": [name for name, _ in STEPS],
        "steps": results,
    }, ensure_ascii=False, indent=2), encoding="utf-8")

save_report()

def execute_step(name):
    if any(record["status"] != "completed" for record in results):
        raise RuntimeError("Uma etapa anterior falhou. Reinicie o kernel e execute Run All.")
    position = len(results)
    if position >= len(STEPS) or name != STEPS[position][0]:
        raise RuntimeError("Etapa fora de ordem. Reinicie o kernel e execute Run All.")
    relative_path = STEPS[position][1]
    source_path = PROJECT_ROOT / relative_path
    log_path = RUN_DIR / f"{name}.log"
    command = (
        [PYTHON_EXECUTABLE, "-u", "-c", NOTEBOOK_RUNNER, str(source_path)]
        if source_path.suffix == ".ipynb"
        else [PYTHON_EXECUTABLE, "-u", "-c", STATE_RUNNER, str(PROJECT_ROOT)]
    )
    record = {"name": name, "source": relative_path, "log": str(log_path), "status": "running"}
    results.append(record)
    save_report()
    started = time.monotonic()
    print(f"Executando {name}: {relative_path}", flush=True)
    environment = dict(os.environ, PYTHONIOENCODING="utf-8", PYTHONUTF8="1", MPLBACKEND="Agg")
    try:
        with log_path.open("w", encoding="utf-8") as log:
            completed = subprocess.run(
                command, cwd=source_path.parent, env=environment,
                stdout=log, stderr=subprocess.STDOUT,
                timeout=STEP_TIMEOUT_SECONDS, check=False,
            )
        record["returncode"] = completed.returncode
        if completed.returncode:
            raise RuntimeError(f"Processo encerrou com código {completed.returncode}.")
    except Exception as error:
        record["status"] = "failed"
        record["error"] = str(error)
        record["elapsed_seconds"] = round(time.monotonic() - started, 2)
        save_report()
        tail = log_path.read_text(encoding="utf-8", errors="replace")[-6000:] if log_path.exists() else ""
        raise RuntimeError(f"Falha em {name}. Consulte {log_path}\n{tail}") from error
    record["status"] = "completed"
    record["elapsed_seconds"] = round(time.monotonic() - started, 2)
    save_report()
    print(f"Concluído em {record['elapsed_seconds']} s. Log: {log_path}", flush=True)

print(f"Python: {PYTHON_EXECUTABLE}\nRelatório: {REPORT_PATH}")


Python: C:\projects\Libellula\.venv\Scripts\python.exe
Relatório: C:\projects\Libellula\data\processed\pipeline_runs\20261011_023812_e71e5324\status.json


## 1. Dataset técnico

Executa Financial Tools sobre as cotações históricas e exporta `financial_tools_datset.csv`, insumo do TFT e das ferramentas de risco.


In [2]:
execute_step("01_financial_tools")


Executando 01_financial_tools: notebooks/financial_tools/financial_tools.ipynb
Concluído em 8.89 s. Log: C:\projects\Libellula\data\processed\pipeline_runs\20261011_023812_e71e5324\01_financial_tools.log


## 2.i. CVaR

Calcula e exporta as features de risco de cauda do CVaR.


In [3]:
execute_step("02_cvar")


Executando 02_cvar: notebooks/risk_envelop/CVaR.ipynb
Concluído em 5.3 s. Log: C:\projects\Libellula\data\processed\pipeline_runs\20261011_023812_e71e5324\02_cvar.log


## 2.ii. EPF

Treina o modelo EPF e exporta o envelope probabilístico. Sua saída também fornece a âncora temporal utilizada pelo EVT.


In [4]:
execute_step("03_epf")


Executando 03_epf: notebooks/risk_envelop/EPF.ipynb
Concluído em 4.84 s. Log: C:\projects\Libellula\data\processed\pipeline_runs\20261011_023812_e71e5324\03_epf.log


## 2.iii. EVT

Executa a análise de valores extremos após EPF, cuja saída é lida pelo notebook EVT.


In [5]:
execute_step("04_evt")


Executando 04_evt: notebooks/risk_envelop/EVT.ipynb
Concluído em 75.75 s. Log: C:\projects\Libellula\data\processed\pipeline_runs\20261011_023812_e71e5324\04_evt.log


## 2.iv. Quantile Forecasting

Treina os modelos de quantis e exporta as features de distribuição e assimetria.


In [6]:
execute_step("05_quantile_forecasting")


Executando 05_quantile_forecasting: notebooks/risk_envelop/QuantileForecasting.ipynb
Concluído em 21.8 s. Log: C:\projects\Libellula\data\processed\pipeline_runs\20261011_023812_e71e5324\05_quantile_forecasting.log


## 2.v. PDFD

Calcula e exporta as probabilidades de quedas para os limiares definidos no notebook.


In [7]:
execute_step("06_pdfd")


Executando 06_pdfd: notebooks/risk_envelop/PDFD.ipynb
Concluído em 4.73 s. Log: C:\projects\Libellula\data\processed\pipeline_runs\20261011_023812_e71e5324\06_pdfd.log


## 3. TFT preditivo

Executa integralmente o notebook TFT, incluindo treinamento, diagnóstico e salvamento dos pesos. Exporta `cotation(t)`, `prediction(t+1)`, `pred_return` e `confidence`, indexadas pelo instante t.


In [8]:
execute_step("07_tft")


Executando 07_tft: notebooks/Predictive_TFT_model/TFT_prdictive_model.ipynb
Concluído em 132.83 s. Log: C:\projects\Libellula\data\processed\pipeline_runs\20261011_023812_e71e5324\07_tft.log


## 4. Concatenação do Envelope de Risco

Executa Concatenation para combinar os cinco datasets de risco em `risk_features.parquet`, com 25 features.


In [9]:
execute_step("08_concatenation")


Executando 08_concatenation: notebooks/risk_envelop/concatenation/Concatenation.ipynb
Concluído em 2.69 s. Log: C:\projects\Libellula\data\processed\pipeline_runs\20261011_023812_e71e5324\08_concatenation.log


## 5. Composição do estado probabilístico

Carrega `rl_state.py` em um processo separado e chama explicitamente `build_rl_state`: executar apenas o arquivo não chama essa função. Une risco e previsões e gera o Parquet e o manifesto de hashes existentes.


In [10]:
execute_step("09_rl_state")


Executando 09_rl_state: src/pipelines/rl_state.py
Concluído em 1.56 s. Log: C:\projects\Libellula\data\processed\pipeline_runs\20261011_023812_e71e5324\09_rl_state.log


## Validação final

Confere o esquema preditivo, as 25 features de risco, os índices temporais, valores finitos e a igualdade entre o estado salvo e a junção dos insumos. Exibe um resumo das nove etapas e a localização dos logs. O estado contém 29 colunas; este pipeline não implementa o ambiente nem o treinamento do agente RL.


In [11]:
import numpy as np
import pandas as pd

if len(results) != len(STEPS) or any(r["status"] != "completed" for r in results):
    raise RuntimeError("O pipeline não concluiu todas as etapas.")

risk = pd.read_parquet(PROJECT_ROOT / "data/processed/combined/risk_features.parquet")
predictions = pd.read_parquet(PROJECT_ROOT / "data/processed/predictions/tft_predictions.parquet")
state = pd.read_parquet(PROJECT_ROOT / "data/processed/rl_state/rl_state.parquet")
assert list(predictions.columns) == ["cotation(t)", "prediction(t+1)", "pred_return", "confidence"]
assert risk.shape[1] == 25 and state.shape[1] == 29
assert len(state) > 0
for frame in [risk, predictions, state]:
    assert frame.index.is_monotonic_increasing and not frame.index.has_duplicates
    assert np.isfinite(frame.to_numpy()).all()
assert pd.DatetimeIndex(risk.index).equals(pd.DatetimeIndex(predictions.index))
assert (predictions["cotation(t)"] > 0).all()
np.testing.assert_allclose(
    predictions["pred_return"],
    (predictions["prediction(t+1)"] - predictions["cotation(t)"]) / predictions["cotation(t)"],
    rtol=1e-6, atol=1e-12,
)
np.testing.assert_allclose(predictions["confidence"], predictions["pred_return"].abs(), rtol=1e-6, atol=1e-12)
expected_state = risk.join(predictions)
expected_state.index = pd.DatetimeIndex(expected_state.index).astype("datetime64[ns]")
expected_state.index.name = "timestamp"
pd.testing.assert_frame_equal(state, expected_state.sort_index())
print(pd.DataFrame(results)[["name", "status", "elapsed_seconds"]].to_string(index=False))
print(f"Validação aprovada: {len(state)} linhas e {state.shape[1]} colunas. Logs: {RUN_DIR}")


                   name    status  elapsed_seconds
     01_financial_tools completed             8.89
                02_cvar completed             5.30
                 03_epf completed             4.84
                 04_evt completed            75.75
05_quantile_forecasting completed            21.80
                06_pdfd completed             4.73
                 07_tft completed           132.83
       08_concatenation completed             2.69
            09_rl_state completed             1.56
Validação aprovada: 205 linhas e 29 colunas. Logs: C:\projects\Libellula\data\processed\pipeline_runs\20261011_023812_e71e5324
